In [0]:
from pyspark.sql.types import *

custom_schema = StructType([
    StructField("date", StringType(), True),
    StructField("actual_mean_temp", IntegerType(), True),
    StructField("actual_min_temp", IntegerType(), True),
    StructField("actual_max_temp", IntegerType(), True),
    StructField("average_min_temp", IntegerType(), True),
    StructField("average_max_temp", IntegerType(), True),
    StructField("record_min_temp", IntegerType(), True),
    StructField("record_max_temp", IntegerType(), True),
    StructField("record_min_temp_year", IntegerType(), True),
    StructField("record_max_temp_year", IntegerType(), True),
    StructField("actual_precipitation", DoubleType(), True),
    StructField("average_precipitation", DoubleType(), True),
    StructField("record_precipitation", DoubleType(), True)
])

weather_df = spark.read.format("csv") \
    .option("header", "true") \
    .schema(custom_schema) \
    .load("/Volumes/workspace/default/source/WS01.csv")

In [0]:
weather_df.show(10)

In [0]:
weather_df.printSchema()

In [0]:
weather_df.write.format("delta").mode("overwrite").saveAsTable("weather_delta")

**Assignment 1:**

In [0]:
#Read Product.csv as a dataframe and save it as a delta table called Product

from pyspark.sql.types import *

#schema defination
product_schema = StructType([
    StructField("productCode", StringType(), True),
    StructField("productName", StringType(), True),
    StructField("productLine", StringType(), True),
    StructField("productScale", StringType(), True),
    StructField("productVendor", StringType(), True),
    StructField("productDescription", StringType(), True),
    StructField("quantityInStock", IntegerType(), True),
    StructField("buyPrice", DoubleType(), True),
    StructField("MSRP", DoubleType(), True)
])

#reading product csv file
product_df = spark.read.format("csv")\
    .option("header","true")\
    .schema(product_schema)\
    .load("/Volumes/workspace/default/source/product.csv")

#writing product table
product_df.write.format("delta").mode("overwrite").saveAsTable("Product")

In [0]:
spark.table("Product").display()

In [0]:
#Read Product_delta.csv as dataframe and perform an upsert transaction on product delta table

from pyspark.sql.types import *
from delta.tables import DeltaTable

#schema defination
Product_delta_schema = StructType([
    StructField("productCode", StringType(), True),
    StructField("productName", StringType(), True),
    StructField("productLine", StringType(), True),
    StructField("productScale", StringType(), True),
    StructField("productVendor", StringType(), True),
    StructField("productDescription", StringType(), True),
    StructField("quantityInStock", IntegerType(), True),
    StructField("buyPrice", DoubleType(), True),
    StructField("MSRP", DoubleType(), True)
])

#reading product csv file
Product_delta_df = spark.read.format("csv")\
    .option("header","true")\
    .schema(product_schema)\
    .load("/Volumes/workspace/default/source/product_delta.csv")

deltaTable = DeltaTable.forName(spark, "Product")

deltaTable.alias("p") \
  .merge(
    Product_delta_df.alias("pd"),
    "p.productCode = pd.productCode"
  ) \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

In [0]:
from pyspark.sql.functions import col
spark.table("Product").filter(col("productCode").isin("S10_1678","S10_1949","S10_9001","S10_9002")).display()

In [0]:
display(spark.sql("DESCRIBE HISTORY Product"))

**Assignment 2:**


In [0]:
# Create a clone of Product table called ProductCL having records before the merge transaction was executed

display(spark.sql("""CREATE TABLE ProductCL CLONE Product VERSION AS OF 0"""))

In [0]:
#Verify that the table does not contain the records from Product_delta file

spark.table("ProductCL") \
     .exceptAll(spark.sql("SELECT * FROM Product VERSION AS OF 0")) \
     .show()

In [0]:
spark.sql("SELECT * FROM Product VERSION AS OF 0") \
     .exceptAll(spark.table("ProductCL")) \
     .show()

In [0]:
spark.table("Product") \
     .select("productCode") \
     .exceptAll(spark.table("ProductCL").select("productCode")) \
     .show()

**Assignment 3:**

In [0]:
# List the transaction history of Product table

display(spark.sql("DESCRIBE HISTORY Product"))

In [0]:

spark.sql("RESTORE TABLE Product TO VERSION AS OF 0")

In [0]:
# Restore Product table to a state before the merge transaction was executed.

spark.sql("RESTORE TABLE Product TO VERSION AS OF 0")

In [0]:
#Check the history again after the Restore command. How does it affect the table history

display(spark.sql("DESCRIBE HISTORY Product"))

**Assignment 4:**

    Load the data from multiple weather stations into Spark delta lake tables. 

In [0]:
#Create a folder and copy the weather history files from above link to a folder in databricks file system.

print("Done")

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Define schema
weather_schema = StructType([
    StructField("date", StringType(), True),
    StructField("actual_mean_temp", IntegerType(), True),
    StructField("actual_min_temp", IntegerType(), True),
    StructField("actual_max_temp", IntegerType(), True),
    StructField("average_min_temp", IntegerType(), True),
    StructField("average_max_temp", IntegerType(), True),
    StructField("record_min_temp", IntegerType(), True),
    StructField("record_max_temp", IntegerType(), True),
    StructField("record_min_temp_year", IntegerType(), True),
    StructField("record_max_temp_year", IntegerType(), True),
    StructField("actual_precipitation", DoubleType(), True),
    StructField("average_precipitation", DoubleType(), True),
    StructField("record_precipitation", DoubleType(), True)
])

# Source path
weather_path = "/Volumes/workspace/default/weather_data"

# Read weather files one at a time using streaming
weather_stream_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .option("maxFilesPerTrigger", 1) \
    .schema(weather_schema) \
    .load(weather_path)

# Transformations
weather_transformed_df = weather_stream_df \
    .drop("record_min_temp_year", "record_max_temp_year") \
    .filter(col("actual_precipitation") > 0) \
    .withColumn(
        "deviation_from_avg",
        abs(col("actual_precipitation") - col("average_precipitation"))
    ) \
    .withColumn("date", to_date(col("date"), "yyyy-M-d")) \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date")))

# Write to Delta table partitioned by year and month
query = weather_transformed_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/weather_data/checkpoints"
    ) \
    .partitionBy("year", "month") \
    .trigger(availableNow=True) \
    .toTable("WeatherData")

In [0]:
spark.table("WeatherData").display()

In [0]:
from pyspark.sql.functions import col, desc

spark.sql("SHOW PARTITIONS WeatherData").orderBy(col("year").desc(), col("month").desc()).show()

**Assignment 5:**

    Compact the files in WeatherData table using OPTIMIZE command. Sort records on Date column during compaction. Note down the number of new files created and number of files deleted.

In [0]:
# Compact the files in WeatherData table using OPTIMIZE command. Sort records on Date column during compaction. Note down the number of new files created and number of files deleted. 

result = spark.sql("OPTIMIZE WeatherData ZORDER BY (date)")
display(result)


# Extract and print key metrics
metrics_row = result.collect()[0]
metrics_dict = metrics_row['metrics']
print(f"Number of files added: {metrics_dict['numFilesAdded']}")
print(f"Number of files removed: {metrics_dict['numFilesRemoved']}")
print(f"Net change in files: {metrics_dict['numFilesAdded'] - metrics_dict['numFilesRemoved']}")